# Ungraded Lab 2: LLM Calls and Crafting Simple Augmented Prompts


Welcome to **LLM Calls and Crafting Simple Augmented Prompts**. In this lab, you'll get hands-on practice with using two essential functions that let you interact with Large Language Models (LLMs). These functions help you both send single prompts to an LLM, and have a back-and-forth conversation. The main aim is to show you how to add extra information to your prompts, making them more detailed and useful. This added context helps the model give you better and more precise responses.

In this lab, you'll learn:

- How to set up and send questions to an LLM for both single questions and conversations.
- How to use additional data to make your prompts richer, improving the model's replies.


# Table of Contents
- [ 1 - Understanding the functions to call LLMs](#1)
  - [ 1.1 `generate_with_single_input`](#1-1)
  - [ 1.2 `generate_with_multiple_input`](#1-2)
- [ 2 - Integrating Data into an LLM Prompt](#2)
  - [ 2.1 Understanding the data structure](#2-1)
  - [ 2.2 Creating the Prompt](#2-2)


In [22]:
from utils import (
    generate_with_single_input, 
    generate_with_multiple_input,
)

<a id='1'></a>
## 1 - Understanding the functions to call LLMs

In this section you will explore the functions that will be used to call LLMs in this course. These functions call the [Anthropic API](https://www.anthropic.com/) to use Claude models. To use these functions, you'll need an Anthropic API key from [console.anthropic.com](https://console.anthropic.com).

<a id='1-1'></a>
### 1.1 `generate_with_single_input`

This function allows you to generate text from a language model based on a single input prompt. For now, let's just focus on a simple call with only a few basic parameters.

#### Parameters:

- `prompt` (str): The input text prompt you want to send to the language model.
- `max_tokens` (int): Maximum tokens to generate in the response.
- `model` (str): The model name. Default is `"claude-3-haiku-20250307"`.
- `anthropic_api_key`: An optional API key for authentication; defaults to `None`. If `None`, the function will use the `ANTHROPIC_API_KEY` environment variable.

In [23]:
# Example call
output = generate_with_single_input(
    prompt="What is the capital of France?"
)

print("Role:", output['role'])
print("Content:", output['content'])

Role: assistant
Content: The capital of France is Paris.


<a id='1-2'></a>
### 1.2 `generate_with_multiple_input`

This function is designed to handle multiple input messages in a conversational context. The input format is a dictionary with two keys:

1. 'role' - the role that the message is being passed (usually "user" or "assistant")
2. 'content' - the prompt

#### Parameters:

- `messages` (List[Dict]): A list of dictionaries, each containing 'role' and 'content' keys to represent each message in the conversation.
- `max_tokens` (int): Determines token limit for the response.
- `model` (str): Model to be used, default is `"claude-3-haiku-20250307"`.
- `anthropic_api_key`: An optional API key for authentication; defaults to `None`. If `None`, the function will use the `ANTHROPIC_API_KEY` environment variable.

In [24]:
# Example call
messages = [
    {'role': 'user', 'content': 'Hello, who won the FIFA world cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user', 'content': 'Who was the captain?'}
]

output = generate_with_multiple_input(
    messages=messages,
    max_tokens=100
)

print("Role:", output['role'])
print("Content:", output['content'])

Role: assistant
Content: Hugo Lloris was the captain of the French national team that won the 2018 FIFA World Cup.


### 1.3 Using the Anthropic SDK Directly

You can also use the Anthropic SDK directly if you want more control over the API calls. In this section, you'll see how to make calls using the `Anthropic` client.

In [25]:
from anthropic import Anthropic
import os

In [26]:
# Create an Anthropic client
# The API key will be loaded from the ANTHROPIC_API_KEY environment variable
client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

To use it, let's consider the same example as before.

In [27]:
messages = [
    {'role': 'user', 'content': 'Hello, who won the FIFA world cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user', 'content': 'Who was the captain?'}
]

In [28]:
response = client.messages.create(
    messages=messages, 
    model="claude-haiku-4-5",
    max_tokens=100
)

In [29]:
import json

print(json.dumps(response.model_dump(), indent=2))

{
  "id": "msg_011CeggPBE7L99Daogpw8Agu",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "Hugo Lloris was the captain of France during the 2018 FIFA World Cup.",
      "type": "text"
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 42,
    "output_tokens": 22,
    "output_tokens_details": null,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}


Notice that the response has several attributes. To access the response content, you can use:

`response.content[0].text`

In [30]:
print(response.content[0].text)

Hugo Lloris was the captain of France during the 2018 FIFA World Cup.


### 1.4 Note on Anthropic Integration

To use the Anthropic API, you'll need an API key from [console.anthropic.com](https://console.anthropic.com). Make sure to:

1. Get an API key from Anthropic's console
2. Set it in the `ANTHROPIC_API_KEY` environment variable (or pass it directly to the functions)

The Anthropic API has rate limits and usage quotas depending on your plan. If you encounter errors:

1. **Rate limit errors (429)**: Wait a moment and retry your request.
2. **Authentication errors**: Verify your API key is correct and set properly.
3. **Invalid request errors**: Check the parameters and message format.

For more information on Anthropic's API, visit the [API documentation](https://docs.anthropic.com/).

<a id='2'></a>
## 2 - Integrating Data into an LLM Prompt

In this section, you will learn how to effectively incorporate data into a prompt before passing it to a Large Language Model (LLM). We will work with a small dataset consisting of JSON files that contain information about houses. It will help you understand how to augment prompts in the context of RAG.

<a id='2-1'></a>
### 2.1 Understanding the data structure

Let's have a quick look in the data structure. It is a tiny dataset of houses. A list containing one dictionary per house.

In [31]:
house_data = [
    {
        "address": "123 Maple Street",
        "city": "Springfield",
        "state": "IL",
        "zip": "62701",
        "bedrooms": 3,
        "bathrooms": 2,
        "square_feet": 1500,
        "price": 230000,
        "year_built": 1998
    },
    {
        "address": "456 Elm Avenue",
        "city": "Shelbyville",
        "state": "TN",
        "zip": "37160",
        "bedrooms": 4,
        "bathrooms": 3,
        "square_feet": 2500,
        "price": 320000,
        "year_built": 2005
    }
]

<a id='2-2'></a>
### 2.2 Creating the Prompt

Let's begin by constructing the prompt. The first step is to design a layout for the data.

In [32]:
# First, let's create a layout for the houses

def house_info_layout(houses):
    # Create an empty string
    layout = ''
    # Iterate over the houses
    for house in houses:
        # For each house, append the information to the string using f-strings
        # The following way using brackets is a good way to make the code readable as in each line you can start a new f-string that will appended to the previous one
        layout += (f"House located at {house['address']}, {house['city']}, {house['state']} {house['zip']} with "
            f"{house['bedrooms']} bedrooms, {house['bathrooms']} bathrooms, "
            f"{house['square_feet']} sq ft area, priced at ${house['price']}, "
            f"built in {house['year_built']}.\n") # Don't forget the new line character at the end!
    return layout

In [39]:
# Check the layout
print(house_info_layout(house_data))

House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.



Now create a function that generates the prompt to be passed to the Language Learning Model (LLM). The function will take a user-provided query and the available housing data as inputs to effectively address the user's query.

In [40]:
def generate_prompt(query, houses):
    # The code made above is modular enough to accept any list of houses, so you could also choose a subset of the dataset.
    # This might be useful in a more complex context where you want to give only some information to the LLM and not the entire data
    houses_layout = house_info_layout(houses)
    # Create a hard-coded prompt. You can use three double quotes (") in this cases, so you don't need to worry too much about using single or double quotes and breaking the code
    PROMPT = f"""
Use the following houses information to answer users queries.
{houses_layout}
Query: {query}    
             """
    return PROMPT

In [41]:
print(generate_prompt("What is the most expensive house?", houses = house_data))


Use the following houses information to answer users queries.
House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.

Query: What is the most expensive house?    
             


Now let's call the LLM!

In [ ]:
query = "What is the most expensive house? And the bigger one?"
# Asking without the augmented prompt
query_without_house_info = generate_with_single_input(prompt = query)
# With house info
enhanced_query = generate_prompt(query, houses = house_data)
query_with_house_info = generate_with_single_input(prompt = enhanced_query)

In [38]:
# Without house info
print(query_without_house_info['content'])

I don't have access to current real estate data, so I can't tell you which specific house is the most expensive or largest right now.

However, I can tell you that:

**Most Expensive Houses** typically include:
- Properties in ultra-wealthy areas (Beverly Hills, Manhattan, London, Dubai)
- Historically, some have sold for $100M-$500M+
- Examples have included mega-mansions owned by billionaires or royalty

**Largest Houses** include:
- Biltmore Estate (North Carolina) - 178,926 sq ft
- Hearst Castle (California) - around 250,000 sq ft of floor space
- Some private estates in the Middle East and Asia

If you're looking for current listings or specific information, I'd recommend checking:
- Real estate websites (Zillow, Luxury Real Estate sites)
- News articles about record-breaking sales
- Architectural or luxury home publications

Are you interested in a particular location or type of property?


In [43]:
# With house info
print(query_with_house_info['content'])

# House Comparison

**Most Expensive House:**
The house at **456 Elm Avenue, Shelbyville, TN 37160** is the most expensive at **$320,000**.

**Bigger House:**
The house at **456 Elm Avenue, Shelbyville, TN 37160** is also the bigger one with **2,500 sq ft** (compared to 1,500 sq ft).

Interestingly, both answers refer to the same property - the Shelbyville house is both more expensive and larger than the Springfield property.


Keep it up! You finished the introductory ungraded lab on how to call LLMs and augment prompts!